# 🥒 Zucchini CNN Regression — sRGB · Combined Lab+HSV · Dual-Head (Ordinal + Regression) · Composite Loss · 5-Fold CV

**Pipeline:** `sRGB nocc TIFF (no filtering) → Lab+HSV combined stack (6ch) → 260305-backbone CNN → dual head (ordinal + regression) → composite loss → 5-fold stratified CV`

This notebook reuses the exact convolutional backbone from `260305_cnn_regression_analysis.ipynb`
(6 conv blocks, 16→32→64→128→256→512 channels, flattened FC head — **not** global-average-pooled),
and extends it the same way `cnn_regression_color_feature_analysis.ipynb` / `ZucchiniRanker_Ultimate_kfold.ipynb`
do: by splitting the final FC head into a **dual head** (an ordinal-regression head + a scalar
regression head) trained jointly with a **composite loss** (ordinal BCE + Huber + pairwise ranking).

Key configuration, per request:

| Setting | Value |
|---|---|
| Input encoding | **sRGB** (standard gamma-encoded TIFF exports — no linearisation) |
| Image size | **128 × 128** |
| Color spaces | **Lab + HSV combined** (stacked as 6 input channels, computed from the sRGB source) |
| Dataset | **NOCC** (non color-corrected) images, **no pixel/colour filtering** applied |
| Cross-validation | **5-fold Stratified K-Fold** |
| Epochs (ceiling) | **1000** (early stopping almost always triggers first) |
| Early-stop patience | **30** (on a composite validation score) |
| LR-reduction patience | **8** (`ReduceLROnPlateau`, same composite score) |
| Heads | Ordinal head (`N_THRESHOLDS` logits) + Regression head (1 scalar, ELU) |
| Loss | `CompositeLoss = 1.0·L_ordinal_BCE + 0.5·L_huber + 0.5·L_ranking` |
| Final prediction | Ensemble: `0.7 · ordinal_score + 0.3 · regression_score` |

Every fold trains a fresh model from scratch; out-of-fold (OOF) predictions are aggregated for the
final reported metrics (R², RMSE, MAE, Spearman ρ, Kendall τ, Acc@0/1/2).


## 0 · Imports & Setup

In [ ]:
import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kendalltau

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              confusion_matrix, classification_report)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch device : {device}')
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
print(f'torch        : {torch.__version__}')


## 1 · Configuration

In [ ]:
# ── EDIT THIS PATH if your dataset moves ────────────────────────────────────
NOCC_ROOT = r"F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01"
# ────────────────────────────────────────────────────────────────────────────

# ── Image / model settings ───────────────────────────────────────────────────
IMG_SIZE       = (128, 128)     # sRGB-encoded input resolution
CNN_DEPTH      = 16             # base channel depth (16->32->64->128->256->512), same as 260305 backbone
BATCH_SIZE     = 32
K_FOLDS        = 5              # stratified cross-validation

# ── Training schedule ────────────────────────────────────────────────────────
EPOCHS           = 1000         # ceiling per fold — early stopping usually stops well before this
LR_INIT          = 5e-4
WEIGHT_DECAY     = 1e-4
LR_FACTOR        = 0.5          # ReduceLROnPlateau factor
PATIENCE_LR      = 8            # LR reduction patience (on composite validation score)
PATIENCE_ES      = 30           # early-stopping patience (on composite validation score)
TARGET_NOISE_STD = 0.15         # Gaussian noise added to training targets only (regularization)

# ── Score / class configuration ──────────────────────────────────────────────
SCORE_FOLDERS = list(range(1, 10))                                # folders 1-9 -> scores 1-9
NUM_CLASSES   = len(SCORE_FOLDERS)                                # 9
IDX_TO_SCORE  = np.arange(1, NUM_CLASSES + 1, dtype=np.float32)   # 1..9
N_THRESHOLDS  = NUM_CLASSES - 1                                   # 8 ordinal thresholds P(score > k+1)

# ── Combined color space (input channels) ────────────────────────────────────
COLOR_SPACES = ['LAB', 'HSV']          # combined, stacked as extra channels derived from the sRGB source
IN_CHANNELS  = 3 * len(COLOR_SPACES)   # 6

CACHE_DIR = 'cnn_cache_labhsv_dualhead'
OUT_DIR   = 'cnn_outputs_labhsv_dualhead'
CKPT_DIR  = 'cnn_checkpoints_labhsv_dualhead'
for d in (CACHE_DIR, OUT_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

def savefig(name, dpi=150):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  saved -> {path}')

print(f'Image size      : {IMG_SIZE}  (sRGB-encoded, 8-bit)')
print(f'Color spaces    : {COLOR_SPACES}  -> {IN_CHANNELS} input channels')
print(f'K-folds         : {K_FOLDS}')
print(f'Epochs (ceiling): {EPOCHS}  | ES patience={PATIENCE_ES}  | LR patience={PATIENCE_LR}')
print(f'Classes         : {NUM_CLASSES}  ({SCORE_FOLDERS})  | ordinal thresholds={N_THRESHOLDS}')


## 2 · Load NOCC Images — sRGB Source, No Filtering

Loads every `nocc`-tagged TIFF from each score folder (1–9). No statistical/Z-score pixel filtering
is applied here — the raw sRGB-encoded image is passed straight through to the color-space stacking
step. If a folder has no `nocc`-tagged file, all files in that folder are used unfiltered as a fallback.

In [ ]:
def load_tif_rgb(path, size):
    '''Load a TIF and return an sRGB-encoded uint8 RGB (H, W, 3) array, resized to `size`.
    No colour-space linearisation is applied here — the source nocc TIFFs are already standard
    gamma-encoded (sRGB-like) exports, so the only conversion is bit-depth normalisation
    (16-bit -> 8-bit) plus channel reordering, matching the reference ablation notebook.'''
    img = tiff.imread(path)
    if img.ndim == 4: img = img[0]
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    if img.ndim == 2:        img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1: img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4: img = img[..., :3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        lo, hi = img.min(), img.max()
        img = (img - lo) / (hi - lo + 1e-8)
        img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)


def collect_data(root, score_folders, img_size):
    '''Walk each score subfolder under root and load ALL nocc TIFFs — no pixel/colour filtering.'''
    rgb_list, label_list, path_list = [], [], []

    print(f'\nSearching root: {root}')
    if not os.path.exists(root):
        raise FileNotFoundError(f'NOCC_ROOT not found: {root}')

    top_items = sorted(os.listdir(root))
    print(f'  Contents ({len(top_items)} items): {top_items[:20]}')

    for folder_num in score_folders:
        candidates = [
            os.path.join(root, str(folder_num)),
            os.path.join(root, f'Score={folder_num:02d}'),
            os.path.join(root, f'Score={folder_num}'),
            os.path.join(root, f'{folder_num:02d}'),
        ]
        folder_path = next((p for p in candidates if os.path.isdir(p)), None)
        if folder_path is None:
            print(f'  [Score {folder_num}] no matching folder found')
            continue

        all_tifs = (glob.glob(os.path.join(folder_path, '*.tif'))
                  + glob.glob(os.path.join(folder_path, '*.TIF'))
                  + glob.glob(os.path.join(folder_path, '*.tiff')))
        all_tifs = list(set(all_tifs))
        nocc_tifs = [p for p in all_tifs if 'nocc' in os.path.basename(p).lower()]
        chosen = nocc_tifs if nocc_tifs else all_tifs
        tag = 'nocc' if nocc_tifs else 'ALL (no nocc tag found — using fallback, no filtering)'
        print(f'  [Score {folder_num}] folder={os.path.basename(folder_path)} '
              f'| all_tifs={len(all_tifs)} | {tag}={len(chosen)}')

        for p in chosen:
            try:
                rgb = load_tif_rgb(p, img_size)
                rgb_list.append(rgb)
                label_list.append(folder_num - 1)   # 0-indexed
                path_list.append(p)
            except Exception as e:
                print(f'    failed {os.path.basename(p)}: {e}')

    if len(rgb_list) == 0:
        raise RuntimeError(f'No images loaded! Check NOCC_ROOT: {root}')

    print(f'\n  Total loaded: {len(rgb_list)} images across {len(set(label_list))} score classes\n')
    return (np.array(rgb_list, dtype=np.uint8),
            np.array(label_list, dtype=np.int32),
            path_list)


X_rgb, y_raw, img_paths = collect_data(NOCC_ROOT, SCORE_FOLDERS, IMG_SIZE)
y = IDX_TO_SCORE[y_raw]     # continuous regression target, 1..9

unique, counts = np.unique(y_raw, return_counts=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(unique + 1, counts, color='seagreen', edgecolor='black')
ax.set_xlabel('Score'); ax.set_ylabel('Count')
ax.set_title('NOCC Image Distribution per Score Class (sRGB source, no filtering)')
ax.set_xticks(unique + 1)
for u, c in zip(unique, counts):
    ax.text(u + 1, c + 0.3, str(c), ha='center', fontsize=9)
plt.tight_layout(); savefig('00_class_distribution.png'); plt.show()

print(f'X_rgb : {X_rgb.shape}   y : {y.shape}   score range: [{y.min():.0f}, {y.max():.0f}]')


## 3 · Combined Color-Space Stack — Lab + HSV (derived from sRGB source)

Each sRGB image is converted to **Lab** and **HSV** (both via OpenCV, operating on the sRGB-encoded
uint8 image) and the two 3-channel results are stacked into a single **6-channel** tensor. This is
the "combined color spaces" input fed to the CNN — no plain RGB channels are included, only the two
derived color-space representations.

In [ ]:
def convert_color_space(img_rgb, space):
    if space == 'LAB': return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    if space == 'HSV': return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    raise ValueError(f'Unknown color space: {space}')


def build_combined_stack(img_rgb):
    '''(H, W, 3) sRGB uint8 -> (H, W, 6) uint8: LAB channels stacked with HSV channels.'''
    return np.concatenate([convert_color_space(img_rgb, cs) for cs in COLOR_SPACES], axis=-1)


t0 = time.time()
X_stack = np.array([build_combined_stack(img) for img in X_rgb], dtype=np.uint8)
print(f'Combined Lab+HSV stack built: {X_stack.shape}  ({time.time()-t0:.1f}s)')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sample_idx = 0
axes[0].imshow(X_rgb[sample_idx]); axes[0].set_title('sRGB (source)'); axes[0].axis('off')
axes[1].imshow(X_stack[sample_idx][:, :, :3]); axes[1].set_title('Lab channels (raw display)'); axes[1].axis('off')
axes[2].imshow(X_stack[sample_idx][:, :, 3:]); axes[2].set_title('HSV channels (raw display)'); axes[2].axis('off')
plt.suptitle('Combined Color-Space Stack — Example Image (6 channels: Lab + HSV)')
plt.tight_layout(); savefig('01_combined_colorspace_example.png'); plt.show()


## 4 · Stratified 5-Fold Cross-Validation Splits

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.arange(len(y_raw)), y_raw))

print(f'StratifiedKFold - {K_FOLDS} folds over {len(y_raw)} images')
for fi, (tr, va) in enumerate(fold_splits):
    cnt = np.bincount(y_raw[va], minlength=NUM_CLASSES)
    print(f'  Fold {fi+1}: train={len(tr):5d}  val={len(va):4d}  val class dist={cnt.tolist()}')


## 5 · Dual-Head CNN — 260305 Backbone + Ordinal & Regression Heads

Identical convolutional backbone to `RegressionCNN` in `260305_cnn_regression_analysis.ipynb` (6 conv
blocks, flattened — not GAP — into a 3-stage FC reduction), but the final single-output layer is
replaced by a **dual head**, exactly as in `cnn_regression_color_feature_analysis.ipynb` /
`ZucchiniRanker_Ultimate_kfold.ipynb`:

- **Ordinal head**: `N_THRESHOLDS` raw logits, `P(score > k+1) = sigmoid(logit_k)`.
- **Regression head**: single scalar with `ELU` activation.

Channel-agnostic via `input_channels` (here `IN_CHANNELS = 6` for the Lab+HSV stack).

In [ ]:
class DualHeadRegressionCNN(nn.Module):
    '''260305_cnn_regression_analysis backbone (6 conv blocks, flattened FC head) with the
    single regression output replaced by a dual head:
        - ord_head : N_THRESHOLDS raw logits -> P(score > k+1) = sigmoid(logit_k)
        - reg_head : single scalar with ELU activation
    Channel-agnostic via `input_channels` (here 6 = Lab + HSV stack).'''
    def __init__(self, input_channels=IN_CHANNELS, img_height=IMG_SIZE[0], img_width=IMG_SIZE[1],
                 depth=CNN_DEPTH, n_thresholds=N_THRESHOLDS, score_min=float(IDX_TO_SCORE.min())):
        super().__init__()
        self.score_min = score_min

        self.conv_layers = nn.Sequential(
            # Block 1: in -> depth
            nn.Conv2d(input_channels, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2: depth -> depth*2
            nn.Conv2d(depth, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.Conv2d(depth * 2, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: depth*2 -> depth*4
            nn.Conv2d(depth * 2, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.Conv2d(depth * 4, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 4: depth*4 -> depth*8
            nn.Conv2d(depth * 4, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.Conv2d(depth * 8, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 5: depth*8 -> depth*16
            nn.Conv2d(depth * 8, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.Conv2d(depth * 16, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 6: depth*16 -> depth*32 (single conv, matching 260305)
            nn.Conv2d(depth * 16, depth * 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # 6 maxpools -> spatial dims shrink by 64x (same formula as 260305/ablation notebooks)
        self.flattened_size = depth * img_height * img_width // 128

        self.shared_fc = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size // 2), nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4), nn.ReLU(),
        )
        head_in = self.flattened_size // 4

        # ── Ordinal head: n_thresholds raw threshold logits ──────────────────────
        self.ord_head = nn.Linear(head_in, n_thresholds)
        _prior_biases = [float(np.log((n_thresholds - k) / (k + 1))) for k in range(n_thresholds)]
        with torch.no_grad():
            self.ord_head.bias.copy_(torch.tensor(_prior_biases, dtype=torch.float32))

        # ── Regression head: single scalar with ELU ──────────────────────────────
        self.reg_head = nn.Sequential(nn.Linear(head_in, 1), nn.ELU())

        print(f'DualHeadRegressionCNN | in_ch={input_channels} | flattened={self.flattened_size} '
              f'-> shared_fc({head_in}) | ord_head({n_thresholds}) + reg_head(1)')

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        shared = self.shared_fc(x)
        ord_logits = self.ord_head(shared)
        reg_out    = self.reg_head(shared)
        return ord_logits, reg_out


_probe = DualHeadRegressionCNN().to(device)
n_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
del _probe


## 6 · Composite Loss (Ordinal BCE + Huber + Ranking) & Ensemble Decoding

$$\mathcal{L}_{total} = 1.0 \cdot \mathcal{L}_{ord} + 0.5 \cdot \mathcal{L}_{huber} + 0.5 \cdot \mathcal{L}_{rank}$$

Ported directly from `cnn_regression_color_feature_analysis.ipynb` / `ZucchiniRanker_Ultimate_kfold.ipynb`,
generalized to `N_THRESHOLDS` (8, since scores run 1–9) and the model's `IDX_TO_SCORE` range.

In [ ]:
class CompositeLoss(nn.Module):
    '''L_total = w_ord * L_ordinal_bce + w_hub * L_huber + w_rank * L_ranking'''
    def __init__(self, w_ord: float = 1.0, w_hub: float = 0.5, w_rank: float = 0.5,
                 delta: float = 0.5, margin: float = 1.0, n_thresholds: int = N_THRESHOLDS,
                 score_min: float = float(IDX_TO_SCORE.min())):
        super().__init__()
        self.w_ord        = w_ord
        self.w_hub        = w_hub
        self.w_rank       = w_rank
        self.margin       = margin
        self.n_thresholds = n_thresholds
        self.score_min    = score_min
        self.huber        = nn.HuberLoss(delta=delta)

    def _ordinal_bce(self, y: torch.Tensor, ord_logits: torch.Tensor) -> torch.Tensor:
        thresholds = self.score_min + torch.arange(1, self.n_thresholds + 1,
                                                     device=y.device, dtype=y.dtype)
        y_ord = (y.unsqueeze(1) > thresholds.unsqueeze(0)).float()
        return F.binary_cross_entropy_with_logits(ord_logits, y_ord)

    def _ranking_loss(self, y: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
        s_i = y.unsqueeze(1);    s_j = y.unsqueeze(0)
        p_i = pred.unsqueeze(1); p_j = pred.unsqueeze(0)
        gap  = s_i - s_j
        mask = (gap.abs() >= 1.0).float()
        loss = torch.clamp(self.margin - gap.sign() * (p_i - p_j), min=0.0)
        return (mask * loss).sum() / (mask.sum() + 1e-8)

    def forward(self, ord_logits: torch.Tensor, reg_out: torch.Tensor, y: torch.Tensor) -> tuple:
        y   = y.squeeze(-1)
        reg = reg_out.squeeze(-1)
        l_ord  = self._ordinal_bce(y, ord_logits)
        l_hub  = self.huber(reg, y)
        l_rank = self._ranking_loss(y, reg)
        total  = self.w_ord * l_ord + self.w_hub * l_hub + self.w_rank * l_rank
        return total, l_ord, l_hub, l_rank


criterion = CompositeLoss(w_ord=1.0, w_hub=0.5, w_rank=0.5, delta=0.5, margin=1.0)


def composite_monitor(r2, rho, tau, acc1, val_loss, mae, rmse):
    '''Weighted blend for early-stop / best-checkpoint selection (R2 is primary).'''
    rho = 0.0 if np.isnan(rho) else rho
    tau = 0.0 if np.isnan(tau) else tau
    return (1.00 * r2
          + 0.50 * rho
          + 0.30 * tau
          + 0.30 * acc1
          - 0.10 * (val_loss / 10.0)
          - 0.20 * (mae / 4.0)
          - 0.20 * (rmse / 4.0))


def predict_score_np(ord_logits: np.ndarray, reg_out: np.ndarray) -> np.ndarray:
    '''Ensemble prediction: 0.7 x ordinal_score + 0.3 x reg_score, clipped to [score_min, score_max].
        ordinal_score = score_min + sum(sigmoid(logit_k)) for k=0..N_THRESHOLDS-1
        reg_score     = clamp(reg_out, score_min, score_max)'''
    score_min, score_max = float(IDX_TO_SCORE.min()), float(IDX_TO_SCORE.max())
    ord_t     = torch.from_numpy(ord_logits)
    ord_score = score_min + torch.sigmoid(ord_t).sum(dim=1).numpy()
    reg_score = np.clip(reg_out.flatten(), score_min, score_max)
    return np.clip(0.7 * ord_score + 0.3 * reg_score, score_min, score_max)


def within_acc(preds: np.ndarray, targets: np.ndarray, gap: int) -> float:
    return float(np.mean(np.abs(np.round(preds) - targets) <= gap))


print('Loss      : CompositeLoss  (1.0*L_ord + 0.5*L_huber + 0.5*L_rank)')
print('ES monitor: 1.00*R2 + 0.50*rho + 0.30*tau + 0.30*Acc@1 - 0.10*(val_loss/10) - 0.20*MAE/4 - 0.20*RMSE/4')
print(f'Optimizer : Adam (lr={LR_INIT}, wd={WEIGHT_DECAY}) - created per fold')
print(f'Scheduler : ReduceLROnPlateau (mode=max, factor={LR_FACTOR}, patience={PATIENCE_LR})')
print(f'Early stop: patience={PATIENCE_ES}, monitor=composite_monitor')


## 7 · Dataset & Augmentation

Random H/V flips + random 90°-multiple rotations (color-safe, geometric only), per-channel
`Normalize(mean=0.5, std=0.5)` generalized to 6 channels, and Gaussian target-noise regularization
(`TARGET_NOISE_STD`) applied to training targets only — same recipe as the 260305/ablation notebooks.

In [ ]:
class CSDualHeadDataset(Dataset):
    '''(H,W,6) uint8 Lab+HSV stack -> augmented, normalized CHW tensor + continuous score target.'''
    def __init__(self, stacks_uint8, labels, augment=False, target_noise_std=0.0):
        self.imgs = torch.from_numpy(stacks_uint8.astype(np.float32) / 255.0).permute(0, 3, 1, 2)
        self.scores = torch.from_numpy(IDX_TO_SCORE[labels]).unsqueeze(1)   # (N,1)
        self.labels = labels
        self.augment = augment
        self.target_noise_std = target_noise_std
        c = self.imgs.shape[1]
        self.normalize = transforms.Normalize(mean=[0.5] * c, std=[0.5] * c)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].clone()
        score = self.scores[idx].clone()

        if self.augment:
            if random.random() < 0.5: img = img.flip(2)   # H flip
            if random.random() < 0.5: img = img.flip(1)   # V flip
            k = random.choice([0, 1, 2, 3])
            if k: img = torch.rot90(img, k=k, dims=[1, 2])

        img = self.normalize(img)

        if self.target_noise_std > 0:
            score = score + torch.randn_like(score) * self.target_noise_std

        return img, score, self.labels[idx]


## 8 · Training Engine (per fold)

`Adam` (`lr=5e-4`, `weight_decay=1e-4`), `ReduceLROnPlateau(mode='max', factor=0.5, patience=8)` and
early stopping (`patience=30`) both tracking the **composite validation score** (not raw loss), and
gradient clipping (`max_norm=5.0`) — same recipe as `cnn_regression_color_feature_analysis.ipynb`.

In [ ]:
def compute_metrics(preds: np.ndarray, targets: np.ndarray) -> dict:
    r2   = r2_score(targets, preds)
    mae  = mean_absolute_error(targets, preds)
    rmse = float(np.sqrt(mean_squared_error(targets, preds)))
    try:    rho, _ = spearmanr(preds, targets)
    except Exception: rho = float('nan')
    try:    tau, _ = kendalltau(preds, targets)
    except Exception: tau = float('nan')
    acc1 = within_acc(preds, targets, gap=1)
    acc0 = within_acc(preds, targets, gap=0)
    return dict(r2=r2, mae=mae, rmse=rmse, rho=rho, tau=tau, acc1=acc1, acc0=acc0)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for imgs, scores, _ in loader:
        imgs, scores = imgs.to(device), scores.to(device)
        optimizer.zero_grad()
        ord_logits, reg_out = model(imgs)
        loss, _, _, _ = criterion(ord_logits, reg_out, scores)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_ord, all_reg, all_targets = [], [], []
    for imgs, scores, _ in loader:
        imgs, scores = imgs.to(device), scores.to(device)
        ord_logits, reg_out = model(imgs)
        loss, _, _, _ = criterion(ord_logits, reg_out, scores)
        total_loss += loss.item() * imgs.size(0)
        all_ord.append(ord_logits.cpu().numpy())
        all_reg.append(reg_out.cpu().numpy())
        all_targets.append(scores.cpu().numpy())

    val_loss = total_loss / len(loader.dataset)
    ord_logits = np.concatenate(all_ord); reg_out = np.concatenate(all_reg)
    targets = np.concatenate(all_targets).flatten()
    preds = predict_score_np(ord_logits, reg_out)
    metrics = compute_metrics(preds, targets)
    return val_loss, metrics, preds, targets


def train_one_fold(fold_idx, train_idx, val_idx, X_stack, y_raw):
    print(f'\n{"="*70}\nFOLD {fold_idx+1}/{K_FOLDS}  (train={len(train_idx)}, val={len(val_idx)})\n{"="*70}')

    train_ds = CSDualHeadDataset(X_stack[train_idx], y_raw[train_idx],
                                  augment=True, target_noise_std=TARGET_NOISE_STD)
    val_ds   = CSDualHeadDataset(X_stack[val_idx], y_raw[val_idx], augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = DualHeadRegressionCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                      factor=LR_FACTOR, patience=PATIENCE_LR)

    history = {'train_loss': [], 'val_loss': [], 'val_composite': [], 'val_r2': [],
               'val_mae': [], 'val_rmse': [], 'lr': []}

    best_score = -np.inf
    best_state = None
    best_preds, best_targets = None, None
    epochs_no_improve = 0
    ckpt_path = os.path.join(CKPT_DIR, f'fold{fold_idx+1}_best.pt')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, metrics, preds, targets = validate_epoch(model, val_loader, criterion)

        composite = composite_monitor(metrics['r2'], metrics['rho'], metrics['tau'],
                                       metrics['acc1'], val_loss, metrics['mae'], metrics['rmse'])
        scheduler.step(composite)
        cur_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_composite'].append(composite)
        history['val_r2'].append(metrics['r2'])
        history['val_mae'].append(metrics['mae'])
        history['val_rmse'].append(metrics['rmse'])
        history['lr'].append(cur_lr)

        improved = composite > best_score
        if improved:
            best_score = composite
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_preds, best_targets = preds.copy(), targets.copy()
            epochs_no_improve = 0
            torch.save(best_state, ckpt_path)
        else:
            epochs_no_improve += 1

        if epoch == 1 or epoch % 10 == 0 or improved:
            flag = ' *' if improved else ''
            print(f'  ep{epoch:4d} | train={train_loss:.4f} val={val_loss:.4f} '
                  f'R2={metrics["r2"]:.4f} MAE={metrics["mae"]:.3f} composite={composite:.4f} '
                  f'lr={cur_lr:.2e} ({time.time()-t0:.1f}s){flag}')

        if epochs_no_improve >= PATIENCE_ES:
            print(f'  Early stopping at epoch {epoch} (no improvement for {PATIENCE_ES} epochs)')
            break

    model.load_state_dict(best_state)
    final_metrics = compute_metrics(best_preds, best_targets)
    final_metrics['composite'] = best_score
    final_metrics['epochs_trained'] = epoch
    print(f'  Fold {fold_idx+1} BEST -> R2={final_metrics["r2"]:.4f} MAE={final_metrics["mae"]:.3f} '
          f'RMSE={final_metrics["rmse"]:.3f} rho={final_metrics["rho"]:.4f} Acc@1={final_metrics["acc1"]:.3f}')

    return dict(model=model, history=history, metrics=final_metrics,
                val_idx=val_idx, preds=best_preds, targets=best_targets, ckpt_path=ckpt_path)


## 9 · Run 5-Fold Cross-Validation Training

⚠️ This trains 5 full models (up to `EPOCHS=1000` each, early-stopped at `patience=30`) — expect a long
run time, especially on CPU. Reduce `EPOCHS`/`K_FOLDS` above for a quick smoke test.

In [ ]:
fold_results = []
oof_pred = np.full(len(y_raw), np.nan, dtype=np.float32)
oof_target = np.full(len(y_raw), np.nan, dtype=np.float32)

t_start = time.time()
for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    result = train_one_fold(fold_idx, train_idx, val_idx, X_stack, y_raw)
    fold_results.append(result)
    oof_pred[val_idx] = result['preds']
    oof_target[val_idx] = result['targets']

print(f'\n{"="*70}\nAll {K_FOLDS} folds finished in {(time.time()-t_start)/60:.1f} min\n{"="*70}')

fold_metrics_list = [r['metrics'] for r in fold_results]
fold_metrics_df = pd.DataFrame(fold_metrics_list)
fold_metrics_df.insert(0, 'fold', range(1, K_FOLDS + 1))
fold_metrics_df.to_csv(os.path.join(OUT_DIR, 'fold_metrics.csv'), index=False)

best_fold_idx = int(fold_metrics_df['composite'].idxmax())
best_fold_score = float(fold_metrics_df.loc[best_fold_idx, 'composite'])
print(f'\nBest fold: {best_fold_idx+1}  (composite={best_fold_score:.4f})')
fold_metrics_df


## 10 · Out-of-Fold Metrics & Fold Summary

In [ ]:
valid_mask = ~np.isnan(oof_pred)
oof_metrics = compute_metrics(oof_pred[valid_mask], oof_target[valid_mask])

print('Out-of-Fold (OOF) Metrics — every image scored exactly once by its held-out fold model')
print('-' * 70)
for k, v in oof_metrics.items():
    print(f'  {k:>6s} : {v:.4f}')

np.save(os.path.join(OUT_DIR, 'oof_pred.npy'), oof_pred)
np.save(os.path.join(OUT_DIR, 'oof_target.npy'), oof_target)
pd.DataFrame([oof_metrics]).to_csv(os.path.join(OUT_DIR, 'oof_metrics.csv'), index=False)

print(f'\nMean per-fold (± std) — R2: {fold_metrics_df["r2"].mean():.4f} ± {fold_metrics_df["r2"].std():.4f}'
      f' | MAE: {fold_metrics_df["mae"].mean():.4f} ± {fold_metrics_df["mae"].std():.4f}'
      f' | RMSE: {fold_metrics_df["rmse"].mean():.4f} ± {fold_metrics_df["rmse"].std():.4f}')


## 11 · Training Curves per Fold

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
metrics_to_plot = ['train_loss', 'val_loss', 'val_composite', 'val_r2', 'val_mae', 'lr']
titles = ['Train Loss', 'Val Loss', 'Val Composite Score', 'Val R2', 'Val MAE', 'Learning Rate']

for ax, key, title in zip(axes.flat, metrics_to_plot, titles):
    for fi, r in enumerate(fold_results):
        ax.plot(r['history'][key], label=f'Fold {fi+1}', alpha=0.85)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    if key == 'lr': ax.set_yscale('log')
    ax.legend(fontsize=8)

plt.suptitle('Training Curves — 5-Fold Cross-Validation (Lab+HSV Dual-Head CNN)')
plt.tight_layout(); savefig('02_training_curves.png'); plt.show()


## 12 · Predicted vs Actual & Residuals (Out-of-Fold)

In [ ]:
p = oof_pred[valid_mask]; t = oof_target[valid_mask]
residuals = p - t

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(t, p, alpha=0.5, s=25, edgecolor='k', linewidth=0.3)
lims = [float(IDX_TO_SCORE.min()) - 0.5, float(IDX_TO_SCORE.max()) + 0.5]
axes[0].plot(lims, lims, 'r--', label='Ideal (y=x)')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual Score'); axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'OOF Predicted vs Actual  (R2={oof_metrics["r2"]:.3f})')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].scatter(t, residuals, alpha=0.5, s=25, edgecolor='k', linewidth=0.3)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Actual Score'); axes[1].set_ylabel('Residual (Pred - Actual)')
axes[1].set_title(f'OOF Residuals  (MAE={oof_metrics["mae"]:.3f}, RMSE={oof_metrics["rmse"]:.3f})')
axes[1].grid(alpha=0.3)

plt.suptitle('Out-of-Fold Prediction Quality — Lab+HSV Dual-Head Composite-Loss CNN')
plt.tight_layout(); savefig('03_oof_pred_vs_actual.png'); plt.show()


## 13 · Confusion Matrix & Classification Metrics (OOF, Rounded Predictions)

Predictions are rounded to the nearest integer score class so we can inspect them as a classification
problem alongside the regression metrics above — raw counts, row-normalized (recall) confusion matrix,
and a full `classification_report` (precision / recall / F1 per class).

In [ ]:
class_labels = [int(s) for s in IDX_TO_SCORE]

pred_rounded = np.clip(np.round(oof_pred[valid_mask]),
                        IDX_TO_SCORE.min(), IDX_TO_SCORE.max()).astype(int)
true_int = np.round(oof_target[valid_mask]).astype(int)

cm = confusion_matrix(true_int, pred_rounded, labels=class_labels)
cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[f'S{s}' for s in class_labels],
            yticklabels=[f'S{s}' for s in class_labels])
axes[0].set_xlabel('Predicted (rounded)'); axes[0].set_ylabel('True')
axes[0].set_title(f'OOF Confusion Matrix — Raw Counts (Acc@0={oof_metrics["acc0"]*100:.1f}%)',
                   fontweight='bold')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=axes[1],
            xticklabels=[f'S{s}' for s in class_labels],
            yticklabels=[f'S{s}' for s in class_labels])
axes[1].set_xlabel('Predicted (rounded)'); axes[1].set_ylabel('True')
axes[1].set_title('OOF Confusion Matrix — Row-Normalized (Recall)', fontweight='bold')

plt.suptitle('Out-of-Fold Confusion Matrix — Lab+HSV Dual-Head Composite-Loss CNN', fontweight='bold')
plt.tight_layout(); savefig('04_confusion_matrix.png'); plt.show()

print('Classification Report (OOF, rounded predictions):')
print('-' * 70)
print(classification_report(true_int, pred_rounded, labels=class_labels,
                             target_names=[f'Score {s}' for s in class_labels],
                             zero_division=0))


## 14 · Prediction Frieze — One Row per True Score Class (max 15 images/row)

Same "frieze" concept as `260305_cnn_regression_analysis.ipynb` / `cnn_regression_color_feature_analysis.ipynb`
(images arranged and titled by predicted score, thumbnails color-coded by error), but laid out with
**one row per true score class** and **capped at 15 images per row** — images within each row are
sorted by predicted score (low → high) and, when a class has more than 15 OOF samples, 15 are evenly
sub-sampled across the sorted order so the full prediction spread per class remains visible.

In [ ]:
N_COLS_FRIEZE = 15   # max images shown per class row

fig, axes = plt.subplots(NUM_CLASSES, N_COLS_FRIEZE,
                          figsize=(N_COLS_FRIEZE * 1.3, NUM_CLASSES * 1.6))

for row, score in enumerate(class_labels):
    cls_idx = np.where(valid_mask & (np.round(oof_target) == score))[0]
    cls_idx = cls_idx[np.argsort(oof_pred[cls_idx])]   # sort low -> high predicted score

    n_show = min(N_COLS_FRIEZE, len(cls_idx))
    if len(cls_idx) > n_show:
        show_idx = cls_idx[np.linspace(0, len(cls_idx) - 1, n_show, dtype=int)]
    else:
        show_idx = cls_idx

    for col in range(N_COLS_FRIEZE):
        ax = axes[row, col]
        if col < n_show:
            idx = show_idx[col]
            pred_s = oof_pred[idx]
            err = abs(pred_s - score)
            color = 'green' if err <= 1.0 else 'red'
            ax.imshow(X_rgb[idx])
            ax.set_title(f'{pred_s:.1f}', fontsize=7, color=color, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values(): spine.set_visible(False)
        else:
            ax.axis('off')

    axes[row, 0].set_ylabel(f'Score {score}', fontsize=10, fontweight='bold',
                             rotation=0, labelpad=32, va='center', ha='right')

plt.suptitle('Prediction Frieze — One Row per True Score Class '
             '(🟢 err≤1 | 🔴 err>1, sorted low→high predicted score, max 15/row)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); savefig('05_prediction_frieze_per_class.png'); plt.show()


## 15 · Summary

In [ ]:
print('=' * 70)
print('SUMMARY — Zucchini CNN Regression (sRGB · Lab+HSV · Dual-Head · Composite Loss · 5-Fold CV)')
print('=' * 70)
print(f'Dataset          : {NOCC_ROOT}')
print(f'                   NOCC (non-color-corrected), no pixel/statistical filtering')
print(f'Images loaded    : {len(y_raw)}  across {NUM_CLASSES} score classes {SCORE_FOLDERS}')
print(f'Input encoding   : sRGB source -> combined Lab + HSV stack ({IN_CHANNELS} channels)')
print(f'Image size       : {IMG_SIZE}')
print(f'Backbone         : 260305 6-block VGG-style CNN (depth={CNN_DEPTH})')
print(f'Heads            : ordinal ({N_THRESHOLDS} thresholds) + regression (ELU scalar)')
print(f'Loss             : CompositeLoss = 1.0*OrdinalBCE + 0.5*Huber(0.5) + 0.5*Ranking(margin=1.0)')
print(f'Final prediction : clip(0.7*ordinal_score + 0.3*regression_score, {IDX_TO_SCORE.min():.0f}, {IDX_TO_SCORE.max():.0f})')
print(f'Cross-validation : StratifiedKFold, K={K_FOLDS}, seed={SEED}')
print(f'Epochs ceiling   : {EPOCHS}  | LR patience={PATIENCE_LR}  | ES patience={PATIENCE_ES}')
print('-' * 70)
print('Out-of-Fold metrics:')
for k, v in oof_metrics.items():
    print(f'  {k:>6s} : {v:.4f}')
print('-' * 70)
print('Per-fold metrics:')
print(fold_metrics_df.to_string(index=False))
print('-' * 70)
print(f'Best fold        : {best_fold_idx+1}  (composite={best_fold_score:.4f})')
print(f'Outputs saved to : {OUT_DIR}/  (figures, fold_metrics.csv, oof_pred.npy, oof_metrics.csv)')
print(f'Checkpoints saved: {CKPT_DIR}/  (fold{{1..{K_FOLDS}}}_best.pt)')
print('=' * 70)
